# Lab: A* Search
## Case Study: Film Production Scheduling

### Scenario

A movie studio is planning the order in which scenes should be filmed.

However, filming cannot happen in arbitrary order:

- Some scenes naturally follow others
- Moving equipment between scenes has a **production cost**
- The production manager wants to **minimize total filming cost**

This problem can be modeled as a **graph**:

- Nodes represent **scenes**
- Edges represent **possible filming transitions**
- Edge weights represent **production cost**

To guide the search efficiently, we also use a **heuristic estimate of remaining production effort** to finish the movie.

Your task is to implement **A\* Search** to determine the optimal filming sequence.

## Input Files

This lab uses the following files.

### scenes.txt
Contains all scenes in the movie.

---

### transitions.txt
Represents possible transitions between scenes.

Format:

CurrentScene  NextScene  ProductionCost

---

### production_goal.txt

Defines the start and goal scenes.

Example:

START OpeningScene  
GOAL CreditsScene  

---

### effort_estimates.txt

Provides heuristic estimates of remaining production effort.

Format:

SceneName  HeuristicValue

The final scene must always have **heuristic value = 0**.

## Learning Objectives

By completing this lab, students will:

- Implement **A\* Search from scratch**
- Understand how heuristics guide search
- Use **priority queues**
- Read structured input from files
- Write results to an output file
- Understand how **g(n) + h(n)** determines search priority

## Program Requirements

Your program must:

- Read input files
- Construct a graph of scenes and transitions
- Implement **A\* search**
- Print results to console
- Write results to `output.txt`

You must implement the following functions:

- `read_scenes()`
- `read_transitions()`
- `read_goal()`
- `read_heuristic()`
- `a_star()`
- `reconstruct_path()`

Do **NOT modify the output writer or the main function**.

## Step 1: Import Required Libraries

In [1]:
import heapq
from collections import defaultdict

## Step 2: Graph Representation

We represent the film scenes as a graph.

Each scene is a node.

Possible filming transitions form edges with a production cost.

Example:

OpeningScene → [(MarketChase, 3), (SecretMeeting, 4)]

In [2]:
class FilmGraph:

    def __init__(self):
        self.nodes = set()
        self.edges = defaultdict(list)

    def add_scene(self, scene):
        self.nodes.add(scene)

    def add_transition(self, scene1, scene2, cost):
        self.edges[scene1].append((scene2, float(cost)))

## Step 3: File Readers

Implement functions to read the input files.

In [3]:
def read_scenes(filename):
    graph = FilmGraph()

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue

            a = line.strip()

            #Populate the graph
            graph.add_scene(a)

    return graph

In [4]:
def read_transitions(graph, filename):

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip() or line.strip().startswith("#"):
                continue

            
            a, b, c = line.strip().split()

            #Populate the graph
            graph.add_transition(a, b, c)

    return graph

In [5]:
def read_goal(filename):
    out = []

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue

            a, b = line.strip().split()

            out.append(b)

    return out[0], out[1]

In [6]:
def read_heuristic(filename):
    heuristic = {}

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue

            a, b = line.strip().split()

            heuristic[a] = int(b)

    return heuristic

## Step 4: Path Reconstruction

Once the goal is found, we reconstruct the path by tracing parent nodes.

In [7]:
def reconstruct_path(came_from, current):
    """
    Reconstruct path from goal back to start using parent pointers.

    create path list and add current node
    follow parent pointers until start node

    return path
    """

    path = []
    
    cur_node = current

    while came_from[cur_node]:
        path.append(cur_node)
        cur_node = came_from[cur_node]

    path.append(cur_node)

    return path[::-1]

## Step 5: Implement A* Search

The evaluation function for A* is:

f(n) = g(n) + h(n)

where

g(n) = cost from start to node  
h(n) = heuristic estimate to goal

In [8]:
def a_star(graph, heuristic, start, goal):

    """
    A* Search pseudocode

    open_set = priority queue
    push start node

    came_from = {}

    g_score[start] = 0

    while open_set not empty:

        current = node with lowest f

        reconstruct path if goal found

        for each neighbor:
            tentative_g = g_score[current] + cost

            if better path:
                update dictionaries
                push neighbor into queue
    """

    open_set = []

    came_from = {}
    g_score = {}

    came_from[start] = None 
    g_score[start] = 0

    heapq.heappush(open_set, (heuristic[start] + g_score[start], start))
    

    while open_set:

        _, current = heapq.heappop(open_set)

        if current == goal:
            path = reconstruct_path(came_from, goal)
            cost = g_score[goal]

            return path, cost

        for neigh in graph.edges[current]:
            node, cost = neigh
            
            if node not in g_score:
                g_score[node] = g_score[current] + cost
                f = g_score[node] + heuristic[node]

                came_from[node] = current
                heapq.heappush(open_set, (f, node))

                continue

            new_g = g_score[current] + cost

            if new_g < g_score[node]:
                g_score[node] = new_g
                f = g_score[node] + heuristic[node]
                
                
                came_from[node] = current
                heapq.heappush(open_set, (f, node))
        
        
    return None, None

## Step 6: Output Writer (Do NOT Modify)

In [9]:
def write_output(path, cost, filename="output.txt"):

    result = "Film Production Plan\n"
    result += "---------------------\n"

    if path:
        result += "Scene Order: " + " -> ".join(path) + "\n"
        result += f"Total Production Cost: {cost}\n"
    else:
        result += "No valid production plan found\n"

    print(result)

    with open(filename, "w") as f:
        f.write(result)

## Step 7: Run the Main Program

Do **NOT modify this code**.

In [10]:
def main():

    graph = read_scenes("scenes.txt")
    read_transitions(graph, "transition.txt")

    start, goal = read_goal("production_goal.txt")

    heuristic = read_heuristic("effort_estimates.txt")

    path, cost = a_star(graph, heuristic, start, goal)

    write_output(path, cost)


if __name__ == "__main__":
    main()

['OpeningScene', 'CityPanorama', 'PoliceInvestigation', 'NewsBroadcast', 'TrainStationRun', 'HelicopterArrival', 'FinalShowdown', 'Aftermath', 'CreditsScene'] 20.0
Film Production Plan
---------------------
Scene Order: OpeningScene -> CityPanorama -> PoliceInvestigation -> NewsBroadcast -> TrainStationRun -> HelicopterArrival -> FinalShowdown -> Aftermath -> CreditsScene
Total Production Cost: 20.0



## Step 8: Discussion Questions

1. Why is A* serach slower than Greedy Best First Search?
2. Why is A* search more memory intensive than Greedy Best First Search?
3. If h(n) == h*(n): would the exploration order or results be different for A* search and GBFS? Why or why not?
4. If h(n) == h*(n): would the solution found by A* search or GBFS be optimal? Why or why not?
5. What property must the heuristic satisfy for A* to guarantee optimal solutions?

In [11]:
# Write your answers below

# Computes and check more paths keeping in mind the cost till now instead of only relying on heuristic
# Computes and check more paths keeping in mind the cost and stores these costs for the paths too
# They'll be same as h(n) will give optimal path so f(n) for other paths will be unoptimal and A* will move to optimal path as the one in GBFS
# Yes they will be optimal as h(n) will guide to optimal path
# Should be admissible and consistent